# Previous Config

## Packages and folders

In [ ]:
!pip install -U ranx sentence-transformers datasets accelerate #xformers
!pip uninstall -y wandb

import argparse
import itertools
import json
import os
import re
import time
import tempfile
import unicodedata
import shutil
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from ranx import Qrels, Run, evaluate
from sentence_transformers import SentenceTransformer, losses, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, InputExample, losses, util, evaluation
from sentence_transformers.training_args import BatchSamplers
from datasets import Dataset
from transformers import EarlyStoppingCallback
from huggingface_hub import list_models, model_info, snapshot_download, login, whoami
from google.colab import userdata

import psutil
import multiprocessing as mp
import traceback
import gc
from multiprocessing import Process, Queue
import signal
import sys

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 3.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 6.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.0/488.0 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.7/285.7 kB 25.6 MB/s eta 0:00:00
   ━━━

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [ ]:
token = userdata.get('HF_TOKEN')
login(token=token)
print(whoami())

{'type': 'user', 'id': '663b5ba4347ccbda6657011c', 'name': 'davidpedidos', 'fullname': 'Garcia', 'isPro': False, 'avatarUrl': '/avatars/c7ec7e4f116a9a483fdbdb95dc009fc2.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'colab_exe', 'role': 'fineGrained', 'createdAt': '2025-12-03T15:52:39.743Z', 'fineGrained': {'canReadGatedRepos': False, 'global': [], 'scoped': [{'entity': {'_id': '663b5ba4347ccbda6657011c', 'type': 'user', 'name': 'davidpedidos'}, 'permissions': []}]}}}}


In [ ]:
# Configuración básica
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
today = time.strftime("%Y-%m-%d")

In [ ]:
def set_directories():
  project_dir = Path(__name__).resolve().parents[1]
  project_dir = project_dir / 'content' / 'TalentCLEF-TaskA'
  date_dir = project_dir.parent / 'output' / today
  date_dir.mkdir(parents=True, exist_ok=True)

  # Crear subdirectorio incremental por ejecución (001, 002, ...)
  exec_dirs = sorted([d for d in date_dir.iterdir() if d.is_dir() and d.name.isdigit() and len(d.name) == 3])
  if exec_dirs and not any(exec_dirs[-1].iterdir()):
      output_dir = exec_dirs[-1]
  else:
      next_id = int(exec_dirs[-1].name) + 1 if exec_dirs else 1
      output_dir = date_dir / f"{next_id:03d}"
      output_dir.mkdir(exist_ok=True)

  return project_dir, output_dir

project_dir, output_dir = set_directories()

In [ ]:
!git clone https://github.com/davidgc14/TalentCLEF-TaskA.git
!cp ./TalentCLEF-TaskA/src/output/ranking_spanish_validation.csv {output_dir.parent.parent}
!rm -r sample_data/

Cloning into 'TalentCLEF-TaskA'...
remote: Enumerating objects: 870, done.
remote: Counting objects: 100% (656/656), done.
remote: Compressing objects: 100% (333/333), done.
remote: Total 870 (delta 381), reused 577 (delta 309), pack-reused 214 (from 1)
Receiving objects: 100% (870/870), 58.18 MiB | 47.78 MiB/s, done.
Resolving deltas: 100% (448/448), done.


## Evaluation functions

In [ ]:
def load_qrels(qrels_path):
    """
    Loads the qrels file (TREC format: q_id, iter, doc_id, rel)
    and converts it to a Qrels object.
    """
    qrels_df = pd.read_csv(qrels_path, sep="\t", header=None,
                           names=["q_id", "iter", "doc_id", "rel"],
                           dtype={"q_id": str, "doc_id": str, "rel":int})

    return Qrels.from_df(qrels_df, q_id_col="q_id", doc_id_col="doc_id", score_col="rel")

def load_run(run_path):
    """
    Loads the run file (TREC format: q_id, Q0, doc_id, rank, score, [tag])
    and converts it to a Run object.
    """
    run_df = pd.read_csv(run_path, sep=r"\s+", header=None)

    # Assign column names based on the number of columns
    if run_df.shape[1] == 5:
        run_df.columns = ["q_id", "Q0", "doc_id", "rank", "score"]
    elif run_df.shape[1] >= 6:
        run_df.columns = ["q_id", "Q0", "doc_id", "rank", "score", "tag"]
    else:
        raise ValueError("The run file does not have the expected format.")

    run_df["q_id"] = run_df.q_id.astype(str)
    run_df["doc_id"] = run_df.doc_id.astype(str)
    return Run.from_df(run_df, q_id_col="q_id", doc_id_col="doc_id", score_col="score")

def evaluate_run(qrels_path, run_path):
    """Evalúa un run y devuelve los resultados como dict."""
    qrels = load_qrels(qrels_path)
    run = load_run(run_path)
    metrics = ["map", "mrr", "ndcg", "precision@5", "precision@10", "precision@100"]
    return evaluate(qrels, run, metrics)

In [ ]:

# ========================
# DATA LOADING AND ENCODING
# ========================

def load_spanish_data(data_dir):
    """Load queries and corpus elements from Spanish data directory."""
    queries_path = data_dir / "queries"
    corpus_elements_path = data_dir / "corpus_elements"

    queries = pd.read_csv(queries_path, sep="\t")
    corpus_elements = pd.read_csv(corpus_elements_path, sep="\t")

    return (
        queries.q_id.to_list(),
        queries.jobtitle.to_list(),
        corpus_elements.c_id.to_list(),
        corpus_elements.jobtitle.to_list(),
    )


def encode_data(model, queries_texts, corpus_texts, model_name, device):
    """Encode queries and corpus using the model."""
    print('Encoding data:', model_name, 'on device:', device)
    query_embeddings = model.encode(queries_texts, convert_to_tensor=True, show_progress_bar=False, device=device)
    corpus_embeddings = model.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=False, device=device)
    return query_embeddings, corpus_embeddings


def calculate_similarities_and_format_results(query_embeddings, corpus_embeddings, queries_ids, corpus_ids, model_name):
    """Calculate cosine similarities and format results in TREC format."""
    print('Calculating similarities and preparing results...')
    similarities = util.cos_sim(query_embeddings, corpus_embeddings).cpu().numpy()

    results = []
    for q_idx, q_id in enumerate(queries_ids):
        sorted_indices = np.argsort(-similarities[q_idx])  # Orden descendente
        for rank, c_idx in enumerate(sorted_indices):
            doc_id = corpus_ids[c_idx]
            score = similarities[q_idx, c_idx]
            results.append(f"{str(q_id)} Q0 {str(doc_id)} {rank+1} {score:.4f} {model_name}")
    return results


def run_evaluation_temp(qrels_path, results, model_name):
    """Run evaluation using temporary file without saving."""
    print('Evaluating Spanish monolingual performance...')

    # Crear archivo temporal
    with tempfile.NamedTemporaryFile(mode='w', suffix='.trec', delete=False, encoding='utf-8') as tmp_file:
        tmp_file.write("\n".join(results))
        tmp_path = tmp_file.name

    try:
        evaluation_results = evaluate_run(qrels_path, tmp_path)
    finally:
        # Eliminar archivo temporal
        os.unlink(tmp_path)

    return evaluation_results


def get_model_name(model):
    """Extract model name from the model object."""
    model_name = model[0].auto_model.config._name_or_path
    return model_name.split("/")[-1]


# ========================
# EVALUATION FUNCTION
# ========================

def spanish_monolingual_evaluation(model, device, source):
    """Evaluate model performance on Spanish monolingual data."""
    data_dir = project_dir / 'data' / source / 'spanish'
    qrels_path = data_dir / "qrels.tsv"

    print('Loading Spanish data...')
    queries_ids, queries_texts, corpus_ids, corpus_texts = load_spanish_data(data_dir)
    model_name = get_model_name(model)

    query_embeddings, corpus_embeddings = encode_data(model, queries_texts, corpus_texts, model_name, device)
    results = calculate_similarities_and_format_results(query_embeddings, corpus_embeddings, queries_ids, corpus_ids, model_name)

    evaluation_results = run_evaluation_temp(qrels_path, results, model_name)
    print('Spanish evaluation completed')
    return evaluation_results


# ========================
# SAVING RESULTS
# ========================

def save_spanish_results(evaluation_results, model_name, nickname, source):
    """Save Spanish monolingual evaluation results to JSON file."""
    print("Saving Spanish evaluation results...")

    json_path = output_dir / "results_spanish_monolingual.json"

    results_data = {
        "metadata": {
            "type": "spanish_monolingual",
            "model_name": model_name,
            "nickname": nickname,
            "source": source,
            "timestamp": today
        },
        "results": evaluation_results
    }

    with open(json_path, "w", encoding="utf-8") as jf:
        json.dump(results_data, jf, indent=2, ensure_ascii=False)

    print(f"Saved Spanish results to {json_path}")


# ========================
# RANKING
# ========================

def update_spanish_ranking(map_score, model_name, nickname, source):
    """Update Spanish-specific ranking CSV file with current execution results."""
    print("Updating Spanish ranking file...")

    ranking_file = output_dir.parent.parent / f"ranking_spanish_{source}.csv"
    execution_id = output_dir.name

    new_record = {
        'timestamp': today,
        'execution_id': execution_id,
        'model_name': model_name,
        'model_alias': nickname,
        'map_es_es': map_score
    }

    if ranking_file.exists():
        df = pd.read_csv(ranking_file)
    else:
        df = pd.DataFrame(columns=['timestamp', 'execution_id', 'model_name', 'model_alias', 'map_es_es'])

    # Double check for existing identical record
    comparison_cols = ['model_name', 'model_alias', 'map_es_es']

    if not df.empty:
        new_record_comparison = {k: new_record.get(k, np.nan) for k in comparison_cols}
        existing_records = df[comparison_cols].to_dict('records')

        for existing in existing_records:
            if all(abs(existing.get(k, np.nan) - new_record_comparison.get(k, np.nan)) < 1e-4
                   if isinstance(new_record_comparison.get(k), (float, int)) and not np.isnan(new_record_comparison.get(k, np.nan))
                   else existing.get(k) == new_record_comparison.get(k)
                   for k in comparison_cols):
                print("Ya existe un registro idéntico en el ranking español. No se agregará el nuevo registro.")
                return

    # Evitar warning de pandas con DataFrame vacío
    if df.empty:
        df = pd.DataFrame([new_record])
    else:
        df = pd.concat([df, pd.DataFrame([new_record])], ignore_index=True)

    df = df.sort_values('map_es_es', ascending=False).reset_index(drop=True)
    df.to_csv(ranking_file, index=False)

    print(f"Ranking español actualizado en {ranking_file}")


# ========================
# MAIN PIPELINE
# ========================

def run_spanish_evaluation(model_name, nickname, device, source):
    """Run complete Spanish monolingual evaluation pipeline."""
    model = SentenceTransformer(model_name, device=device)

    evaluation_results = spanish_monolingual_evaluation(model, device, source)
    save_spanish_results(evaluation_results, model_name, nickname, source)

    map_score = evaluation_results.get('map', np.nan)
    update_spanish_ranking(map_score, model_name, nickname, source)

    print(f"\n{'='*50}")
    print(f"Evaluación completada para {model_name}")
    print(f"MAP español-español: {map_score:.4f}")
    print(f"{'='*50}\n\n")

# Training path

## Config

In [ ]:
MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'
BATCH_SIZE = 64
EPOCHS = 30
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Rutas de salida para el entrenamiento Hugging Face
model_output_path = output_dir / 'finetuned_models' / MODEL_NAME
best_model_path = model_output_path / 'best_model'
logs_path = model_output_path / 'logs'

print(f"Directorio de proyecto: {project_dir}")
print(f"Guardando outputs en: {model_output_path}")

Directorio de proyecto: /content/TalentCLEF-TaskA
Guardando outputs en: /content/output/2025-11-26/001/finetuned_models/paraphrase-multilingual-mpnet-base-v2


## Data preparation

### Training data

In [ ]:
def normalize_text(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = unicodedata.normalize('NFC', text)
    # Reemplazar puntuación por espacio para no pegar palabras
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
training_file = project_dir / 'data' / 'training' / 'spanish' / 'taskA_training_es.tsv'

train_df = pd.read_csv(training_file, sep='\t', header=None)
train_df.columns = ['family_id', 'id', 'jobtitle_1', 'jobtitle_2']

In [ ]:
expanded = []
for _, row in train_df.iterrows():
    jt1 = row['jobtitle_1']
    jt2 = row['jobtitle_2']

    # Obtener opciones para cada columna
    opts1 = [x.strip() for x in jt1.split('/')]
    opts2 = [x.strip() for x in jt2.split('/')]

    # Crear combinaciones A × B
    for a, b in itertools.product(opts1, opts2):
        if a != b:
            expanded.append({
                't1': a,
                't2': b
            })

df_expanded = pd.DataFrame(expanded)

In [ ]:
# Normalización
df_expanded['t1'] = df_expanded['t1'].apply(normalize_text)
df_expanded['t2'] = df_expanded['t2'].apply(normalize_text)

# Eliminar duplicados (considerando pares desordenados)
# Creamos una columna temporal 'key' para filtrar
df_expanded['key'] = df_expanded.apply(lambda x: frozenset([x['t1'], x['t2']]), axis=1)
df_unique = df_expanded.drop_duplicates(subset='key').drop(columns=['key'])

# Filtrar vacíos
df_unique = df_unique[(df_unique['t1'] != "") & (df_unique['t2'] != "")]

# Convertir las columnas a string ANTES de crear el Dataset
df_unique['t1'] = df_unique['t1'].astype(str)
df_unique['t2'] = df_unique['t2'].astype(str)

# Resetear índice y crear el Dataset sin preservar el índice de pandas
df_unique = df_unique.reset_index(drop=True)
train_df_for_ds = df_unique[['t1', 't2']].rename(columns={'t1': 'sentence_1', 't2': 'sentence_2'})

In [ ]:
train_dataset = Dataset.from_pandas(train_df_for_ds, preserve_index=False)
print(f"Total de parejas en entrenamiento: {len(train_dataset)}")

Total de parejas en entrenamiento: 17855


### Validation data

In [ ]:
validation_dir = project_dir / 'data' / 'validation' / 'spanish'

corpus_val = pd.read_csv(validation_dir / 'corpus_elements', sep='\t')
queries_val = pd.read_csv(validation_dir / 'queries', sep='\t')
qrels_val = pd.read_csv(validation_dir / 'qrels.tsv', sep='\t', header=None)
qrels_val.columns = ['q_id', 'iter', 'doc_id', 'rel']

In [ ]:
# Diccionarios normalizados
corpus_dict = {str(row.iloc[0]): normalize_text(row.iloc[1]) for _, row in corpus_val.iterrows()}
queries_dict = {str(row.iloc[0]): normalize_text(row.iloc[1]) for _, row in queries_val.iterrows()}

# Construir dataset de pares positivos para calcular la Loss de validación
val_pairs = []
for _, row in qrels_val.iterrows():
    q_id = str(row.iloc[0])
    c_id = str(row.iloc[2])
    score = int(row.iloc[3])

    if score > 0 and q_id in queries_dict and c_id in corpus_dict:
        val_pairs.append({
            'sentence_1': queries_dict[q_id],
            'sentence_2': corpus_dict[c_id]
        })

In [ ]:
val_dataset = Dataset.from_list(val_pairs)
print(f"Pares de validación para Early Stopping: {len(val_dataset)}")

Pares de validación para Early Stopping: 7579


## Model preparation

In [ ]:
# Cargar modelo
model = SentenceTransformer(MODEL_NAME, device=DEVICE)

# Definir función de pérdida
# MultipleNegativesRankingLoss es estándar para pares (Anchor, Positive)
train_loss = losses.MultipleNegativesRankingLoss(model=model)

# Definir argumentos de entrenamiento (usando la clase nativa de SentenceTransformers/HF)
args = SentenceTransformerTrainingArguments(
    output_dir=str(model_output_path),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=(DEVICE == 'cuda'),  # Usar precisión mixta si hay GPU
    eval_strategy="epoch",    # Evaluar al final de cada época
    save_strategy="epoch",    # Guardar checkpoint al final de cada época
    load_best_model_at_end=True, # Cargar el mejor modelo al terminar (CRUCIAL para Early Stopping)
    metric_for_best_model="eval_loss",
    save_total_limit=2,       # No llenar el disco con checkpoints
    logging_steps=50,
    batch_sampler=BatchSamplers.NO_DUPLICATES # Evitar duplicados en batch para MNRLoss
)

early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=5,  # Número de épocas sin mejora antes de detener
    early_stopping_threshold=0.001 # Umbral mínimo de mejora
)

# Inicializar Trainer con EarlyStoppingCallback
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=train_loss,
    callbacks=[early_stopping_callback]
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

## Training model

In [ ]:
print("\n--- Comenzando entrenamiento... ---")
trainer.train()
print("\n--- Entrenamiento finalizado ---")


--- Comenzando entrenamiento... ---


Epoch,Training Loss,Validation Loss
1,0.471600,2.353265
2,0.224400,2.603937
3,0.157300,2.700533
4,0.128800,2.809389
5,0.070300,2.844799
6,0.061200,2.923309



--- Entrenamiento finalizado ---


In [ ]:
print(f"Guardando mejor modelo en: {best_model_path}")
model.save(str(best_model_path))

final_model = SentenceTransformer(str(best_model_path), device=DEVICE)

Guardando mejor modelo en: /content/output/2025-11-26/001/finetuned_models/paraphrase-multilingual-mpnet-base-v2/best_model


## Final evaluation

In [ ]:

# Ejecutar tu evaluación personalizada (si existe la función en el contexto)
# Asumimos que run_spanish_evaluation estaba definida o se importó.
# Si no, simplemente imprimimos confirmación.

print("\nModelo cargado y listo para inferencia.")
# Ejemplo de uso rápido:
# embeddings = final_model.encode(["ingeniero de datos", "data engineer"])
# sim = util.cos_sim(embeddings[0], embeddings[1])
# print(f"Similitud de prueba: {sim.item()}")


Modelo cargado y listo para inferencia.


In [ ]:
spanish_monolingual_evaluation(final_model, DEVICE, 'validation')

Loading Spanish data...
Encoding data: best_model on device: cuda


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/146 [00:00<?, ?it/s]

Calculating similarities and preparing results...
Evaluating Spanish monolingual performance...


/usr/local/lib/python3.12/dist-packages/ranx/metrics/average_precision.py:49: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  scores[i] = _average_precision(qrels[i], run[i], k, rel_lvl)


Spanish evaluation completed


{'map': np.float64(0.45862232062435015),
 'mrr': np.float64(0.5563063063063063),
 'ndcg': np.float64(0.745120995023976),
 'precision@5': np.float64(0.6605405405405405),
 'precision@10': np.float64(0.6545945945945946),
 'precision@100': np.float64(0.24421621621621625)}

In [ ]:
#comprimir carpeta del best_model
shutil.make_archive(best_model_path, 'zip', project_dir)

'/content/output/2025-11-26/001/finetuned_models/paraphrase-multilingual-mpnet-base-v2/best_model.zip'

In [ ]:
best_model_path

PosixPath('/content/output/2025-11-26/001/finetuned_models/paraphrase-multilingual-mpnet-base-v2/best_model')

# Massive Experimentation

## Hugging face iteration

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
source = 'validation'

models = list(list_models(
    pipeline_tag="sentence-similarity",
    library=["sentence-transformers", "transformers"],
    language="es"
))

models = sorted(models, key=lambda x: x.id)

NameError: name 'torch' is not defined

In [ ]:
df = pd.DataFrame(columns=['model_name', 'num_params', 'languages', 'map_es_es', 'mrr', 'ndcg', 'p@5', 'p@10', 'p@100'])
errors = []
for m in models:
  info = model_info(m.id)
  langs = info.cardData.get("languages") or info.cardData.get("language")
  mode = 'multilingual' if len(langs) > 1 else 'monolingual'

  try:
    model = SentenceTransformer(m.id, device=device, trust_remote_code=True)
    evaluation_results = spanish_monolingual_evaluation(model, device, source)
  except:
    errors.append(m.id)
    continue

  num_params = sum(p.numel() for p in model.parameters())

  df = pd.concat([df, pd.DataFrame([{
        'model_name': m.id,
        'num_params': num_params,
        'languages': mode,
        'map_es_es': evaluation_results.get('map', np.nan),
        'mrr': evaluation_results.get('mrr', np.nan),
        'ndcg': evaluation_results.get('ndcg', np.nan),
        'p@5': evaluation_results.get('precision@5', np.nan),
        'p@10': evaluation_results.get('precision@10', np.nan),
        'p@100': evaluation_results.get('precision@100', np.nan)
  }])], ignore_index=True)


df = df.sort_values('map_es_es', ascending=False).reset_index(drop=True)

In [ ]:
df

In [ ]:
df.to_csv(output_dir / 'hf_evaluation_results.csv', index=False)

# los modelos que no se han podido ejecutar es porque son cuantizaciones de modelos mayores que si se han ejecutado
with open(output_dir / "hf_errors.txt", "w") as archivo:
    for item in errors:
        archivo.write(f"{item}\n")

## Other models from MTEB leaderboard

In [ ]:
model_name = 'McGill-NLP/LLM2Vec-Mistral-7B-Instruct-v2-mntp-supervised'

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
source = 'validation'

model = SentenceTransformer(model_name, device=device, trust_remote_code=True)
evaluation_results = spanish_monolingual_evaluation(model, device, source)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
info = model_info(model_name)
langs = info.cardData.get("languages") or info.cardData.get("language")
mode = 'multilingual' if len(langs) > 1 else 'monolingual'
num_params = sum(p.numel() for p in model.parameters())

print(f"Model name: {model_name}")
print(f"Languages: {mode}")
print(f"Number of parameters: {num_params}")

print(evaluation_results)

model

## MTEB models itteration

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
source = 'validation'

with open(project_dir / 'models' / 'results' / 'mteb' / 'mteb_models_list.txt', 'r') as file:
    models = file.read().splitlines()

models = sorted(models, reverse=True)

df = pd.DataFrame(columns=['model_name', 'num_params', 'languages', 'map_es_es', 'mrr', 'ndcg', 'p@5', 'p@10', 'p@100'])
errors = []

In [ ]:
df = pd.read_csv('/content/mteb_evaluation_results_checkpoint.csv')

with open('/content/mteb_errors_checkpoint.txt', "r") as archivo:
    errors = [line.strip() for line in archivo.readlines()]

In [ ]:
df

,model_name,num_params,languages,map_es_es,mrr,ndcg,p@5,p@10,p@100
0,thenlper/gte-small,33360000,monolingual,0.304450,0.535157,0.645663,0.605405,0.537297,0.171730
1,thenlper/gte-large,335141888,monolingual,0.341373,0.544084,0.672224,0.614054,0.569189,0.187946
2,thenlper/gte-base,109482240,monolingual,0.319907,0.546937,0.657480,0.602162,0.549189,0.179405
3,sentence-transformers/all-mpnet-base-v2,109486464,multilingual,0.263311,0.538492,0.622412,0.561081,0.493514,0.150541
4,sentence-transformers/all-MiniLM-L6-v2,22713216,multilingual,0.283830,0.539444,0.630327,0.572973,0.525946,0.158216
5,sentence-transformers/all-MiniLM-L12-v2,33360000,multilingual,0.291990,0.534586,0.636105,0.576216,0.521622,0.166378
6,sdadas/mmlw-roberta-large,434961408,multilingual,0.267101,0.552162,0.627704,0.569730,0.497297,0.150595
7,sentence-transformers/paraphrase-multilingual-...,117653760,multilingual,0.376560,0.541937,0.697857,0.607568,0.572973,0.211784
8,sdadas/mmlw-e5-large,559890432,multilingual,0.327969,0.548174,0.665108,0.588108,0.533514,0.179838
9,sdadas/mmlw-e5-base,278043648,multilingual,0.308906,0.530482,0.652663,0.582703,0.534595,0.171730


In [ ]:
errors

['sentence-transformers/static-similarity-mrl-multilingual-v1',
 'parasail-ai/GritLM-7B-vllm']

In [ ]:
for index, model_name in enumerate(models):

  if model_name in df.model_name.values or model_name in errors:
    continue

  if index == 0:
    print('Comenzamos desde el principio')
  else:
    print(f'Continuamos con iteracion {index}.')

  print(f'Cargando modelo {model_name}')

  try:
    info = model_info(model_name)
    langs = info.cardData.get("languages") or info.cardData.get("language")
    mode = 'multilingual' if len(langs) > 1 else 'monolingual'
  except:
    mode = 'unknown'

  try:
    # descarga local para separar los dos procesos de descarga y carga
    local_dir = snapshot_download(repo_id=model_name, cache_dir="/content/hf_cache", use_auth_token=True, resume_download=True, force_download=False, local_files_only=False)
    model = SentenceTransformer(local_dir, device=device, trust_remote_code=True)
    evaluation_results = spanish_monolingual_evaluation(model, device, source)
  except Exception as e:
    print(f'Error al cargar modelo {model_name}: {e}')
    errors.append(model_name)
    continue

  num_params = sum(p.numel() for p in model.parameters())

  df = pd.concat([df, pd.DataFrame([{
        'model_name': model_name,
        'num_params': num_params,
        'languages': mode,
        'map_es_es': evaluation_results.get('map', np.nan),
        'mrr': evaluation_results.get('mrr', np.nan),
        'ndcg': evaluation_results.get('ndcg', np.nan),
        'p@5': evaluation_results.get('precision@5', np.nan),
        'p@10': evaluation_results.get('precision@10', np.nan),
        'p@100': evaluation_results.get('precision@100', np.nan)
  }])], ignore_index=True)


  if index % 1 == 0:
    df.to_csv(output_dir / 'mteb_evaluation_results_checkpoint.csv', index=False)
    with open(output_dir / "mteb_errors_checkpoint.txt", "w") as archivo:
        for item in errors:
            archivo.write(f"{item}\n")

    print('==========================================')
    print(f'Checkpoint en iteracion {index}')
    print('==========================================')

Continuamos con iteracion 12.
Cargando modelo nvidia/NV-Embed-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
df = df.sort_values('map_es_es', ascending=False).reset_index(drop=True)
df.to_csv(output_dir / 'mteb_evaluation_results.csv', index=False)

# los modelos que no se han podido ejecutar es porque son cuantizaciones de modelos mayores que si se han ejecutado
with open(output_dir / "mteb_errors.txt", "w") as archivo:
    for item in errors:
        archivo.write(f"{item}\n")

In [ ]:
df

In [ ]:
errors

## MTEB Models Iteration con control de RAM

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
source = 'validation'

with open(project_dir / 'models' / 'results' / 'mteb' / 'mteb_models_list.txt', 'r') as file:
    models = file.read().splitlines()

models = sorted(models, reverse=True)

df = pd.DataFrame(columns=['model_name', 'num_params', 'languages', 'map_es_es', 'mrr', 'ndcg', 'p@5', 'p@10', 'p@100'])
reported_errors = pd.DataFrame(columns=['model_name', 'error'])

errors = []

In [ ]:
df = pd.read_csv('/content/mteb_evaluation_results_checkpoint.csv')
reported_errors = pd.read_csv('/content/mteb_reported_errors_checkpoint.csv')

combined = pd.read_csv(project_dir / 'models' / 'results' / 'temp_combined_evaluation.csv')

with open('/content/mteb_errors_checkpoint.txt', "r") as archivo:
    errors = [line.strip() for line in archivo.readlines()]

In [ ]:
df

,model_name,num_params,languages,map_es_es,mrr,ndcg,p@5,p@10,p@100
0,nvidia/llama-embed-nemotron-8b,7504924672,multilingual,0.226806,0.525526,0.604436,0.496216,0.418378,0.135405
1,thenlper/gte-small,33360000,monolingual,0.304450,0.535157,0.645663,0.605405,0.537297,0.171730
2,thenlper/gte-large,335141888,monolingual,0.341373,0.544084,0.672224,0.614054,0.569189,0.187946
3,thenlper/gte-base,109482240,monolingual,0.319907,0.546937,0.657480,0.602162,0.549189,0.179405
4,sentence-transformers/all-mpnet-base-v2,109486464,multilingual,0.263311,0.538492,0.622412,0.561081,0.493514,0.150541
5,sentence-transformers/all-MiniLM-L6-v2,22713216,multilingual,0.283830,0.539444,0.630327,0.572973,0.525946,0.158216
6,sentence-transformers/all-MiniLM-L12-v2,33360000,multilingual,0.291990,0.534586,0.636105,0.576216,0.521622,0.166378
7,sdadas/mmlw-roberta-large,434961408,multilingual,0.267101,0.552162,0.627704,0.569730,0.497297,0.150595
8,sentence-transformers/paraphrase-multilingual-...,117653760,multilingual,0.376560,0.541937,0.697857,0.607568,0.572973,0.211784
9,sdadas/mmlw-e5-large,559890432,multilingual,0.327969,0.548174,0.665108,0.588108,0.533514,0.179838


In [ ]:
errors

['sentence-transformers/static-similarity-mrl-multilingual-v1',
 'parasail-ai/GritLM-7B-vllm',
 'nvidia/NV-Embed-v2',
 '',
 'nvidia/NV-Embed-v1',
 'nan',
 'minishlab/potion-multilingual-128M',
 'jinaai/jina-embeddings-v4',
 'jinaai/jina-embeddings-v3',
 'intfloat/multilingual-e5-small',
 'intfloat/multilingual-e5-large-instruct',
 'intfloat/multilingual-e5-large',
 'intfloat/multilingual-e5-base',
 'intfloat/e5-mistral-7b-instruct',
 'intfloat/e5-large-v2',
 'intfloat/e5-base-v2',
 'infly/inf-retriever-v1',
 'google/embeddinggemma-300m',
 'gandolfi/bge-m3-custom-fr-Q4_K_M-GGUF',
 'bartowski/granite-embedding-107m-multilingual-GGUF',
 'avsolatorio/NoInstruct-small-Embedding-v0',
 'Snowflake/snowflake-arctic-embed-m-v2.0',
 'Salesforce/SFR-Embedding-Mistral',
 'Salesforce/SFR-Embedding-2_R',
 'Qwen/Qwen3-Embedding-8B',
 'Qwen/Qwen3-Embedding-4B',
 'NovaSearch/stella_en_400M_v5',
 'NovaSearch/stella_en_1.5B_v5',
 'NovaSearch/jasper_en_vision_language_v1']

In [ ]:
# Configuración
RAM_THRESHOLD = 90  # Ajusta según tu entorno (80-85 para Colab)
VRAM_THRESHOLD = 95  # Umbral para VRAM (puede ser más alto que RAM)
RAM_CHECK_INTERVAL = 0.3  # Verificar cada 0.5 segundos

def get_ram_usage():
    """Retorna el porcentaje de RAM utilizado"""
    return psutil.virtual_memory().percent

def get_vram_usage():
    """Retorna el porcentaje de VRAM utilizado"""
    if torch.cuda.is_available():
        total_vram = torch.cuda.get_device_properties(0).total_memory
        allocated_vram = torch.cuda.memory_allocated(0)
        return (allocated_vram / total_vram) * 100
    return 0

def get_vram_gb():
    """Retorna VRAM usado en GB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated(0) / 1024**3
    return 0

def clear_memory():
    """Libera memoria de forma agresiva"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def load_model_with_monitoring(model_name, local_dir, device, ram_threshold=85, vram_threshold=90, check_interval=2.0):
    """
    Carga un modelo monitoreando RAM y VRAM periódicamente durante la carga.

    Args:
        model_name: Nombre del modelo
        local_dir: Directorio local del modelo
        device: Dispositivo (cuda/cpu)
        ram_threshold: Umbral de RAM en porcentaje
        vram_threshold: Umbral de VRAM en porcentaje
        check_interval: Intervalo de verificación en segundos

    Returns:
        tuple: (model, num_params, error_message)
    """
    try:
        print(f'   📦 Iniciando carga del modelo...')

        # Verificar memoria antes de empezar
        ram_initial = get_ram_usage()
        vram_initial = get_vram_usage()
        print(f'   📊 Pre-carga -> RAM: {ram_initial:.1f}% | VRAM: {vram_initial:.1f}%')

        if ram_initial > ram_threshold:
            return None, None, f"RAM_TOO_HIGH_BEFORE_LOAD", ram_initial, vram_initial

        if vram_initial > vram_threshold:
            return None, None, f"VRAM_TOO_HIGH_BEFORE_LOAD", ram_initial, vram_initial

        # Cargar el modelo (SIN subprocess, para evitar problemas de serialización)
        start_time = time.time()
        model = SentenceTransformer(local_dir, device=device, trust_remote_code=True, model_kwargs={"torch_dtype": torch.float16, "low_cpu_mem_usage": True})
        load_time = time.time() - start_time

        # Verificar memoria después de cargar
        ram_after = get_ram_usage()
        vram_after = get_vram_usage()
        vram_gb = get_vram_gb()

        print(f'   ✓ Modelo cargado en {load_time:.1f}s')
        print(f'   📊 Post-carga -> RAM: {ram_after:.1f}% | VRAM: {vram_after:.1f}% ({vram_gb:.2f} GB)')

        # Verificar si se excedieron los umbrales DESPUÉS de cargar
        if ram_after > ram_threshold:
            print(f'   ⚠️  RAM excedió umbral después de cargar: {ram_after:.1f}% > {ram_threshold}%')
            del model
            clear_memory()
            return None, None, f"RAM_EXCEEDED_AFTER_LOAD", ram_after, vram_after

        if vram_after > vram_threshold:
            print(f'   ⚠️  VRAM excedió umbral después de cargar: {vram_after:.1f}% > {vram_threshold}%')
            del model
            clear_memory()
            return None, None, f"VRAM_EXCEEDED_AFTER_LOAD", ram_after, vram_after

        # Obtener número de parámetros
        num_params = sum(p.numel() for p in model.parameters())

        return model, num_params, None, ram_after, vram_after

    except RuntimeError as e:
        if "out of memory" in str(e).lower() or "cuda" in str(e).lower():
            ram_curr = get_ram_usage()
            vram_curr = get_vram_usage()
            print(f'   ❌ Error de memoria CUDA: {e}')
            print(f'   📊 Al fallar -> RAM: {ram_curr:.1f}% | VRAM: {vram_curr:.1f}%')
            clear_memory()
            return None, None, f"CUDA_OUT_OF_MEMORY", ram_curr, vram_curr
        else:
            raise

    except Exception as e:
        ram_curr = get_ram_usage()
        vram_curr = get_vram_usage()
        print(f'   ❌ Error inesperado: {str(e)[:100]}')
        return None, None, str(e)[:200], ram_curr, vram_curr

In [ ]:
cache_dir = "/content/hf_cache"

print(f'🚀 Iniciando evaluación con control de RAM y VRAM')
print(f'📊 Umbral RAM: {RAM_THRESHOLD}% | Umbral VRAM: {VRAM_THRESHOLD}%')
print(f'💾 RAM inicial: {get_ram_usage():.1f}%')
if torch.cuda.is_available():
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'🎮 CUDA disponible: {torch.cuda.get_device_name(0)}')
    print(f'🎮 VRAM total: {total_vram_gb:.1f} GB')
    print(f'🎮 VRAM inicial: {get_vram_usage():.1f}% ({get_vram_gb():.2f} GB)')
print('=' * 60)

for index, model_name in enumerate(models):
    if model_name in df.model_name.values or model_name in errors or model_name in combined.model_name.values:
        continue

    if index == 0:
        print('Comenzamos desde el principio')
    else:
        print(f'Continuamos con iteracion {index}.')

    # Estado de memoria antes de empezar
    ram_before = get_ram_usage()
    vram_before = get_vram_usage()
    vram_gb_before = get_vram_gb()
    print(f'📊 Estado inicial -> RAM: {ram_before:.1f}% | VRAM: {vram_before:.1f}% ({vram_gb_before:.2f} GB)')

    # Verificación preventiva
    if ram_before > RAM_THRESHOLD or vram_before > VRAM_THRESHOLD:
        print(f'⚠️  Memoria demasiado alta antes de cargar - Limpiando...')
        clear_memory()
        time.sleep(1)  # Dar tiempo a que se libere
        ram_after_clean = get_ram_usage()
        vram_after_clean = get_vram_usage()
        print(f'📊 Después de limpieza -> RAM: {ram_after_clean:.1f}% | VRAM: {vram_after_clean:.1f}%')

        if ram_after_clean > RAM_THRESHOLD:
            print(f'❌ No se pudo liberar suficiente RAM ({ram_after_clean:.1f}% > {RAM_THRESHOLD}%) - Saltando {model_name}')
            errors.append(model_name)
            reported_errors = pd.concat([reported_errors, pd.DataFrame([{
                'model_name': model_name,
                'error': f'RAM_HIGH_BEFORE_LOAD ({ram_after_clean:.1f}%)'
            }])], ignore_index=True)
            continue

        if vram_after_clean > VRAM_THRESHOLD:
            print(f'❌ No se pudo liberar suficiente VRAM ({vram_after_clean:.1f}% > {VRAM_THRESHOLD}%) - Saltando {model_name}')
            errors.append(model_name)
            reported_errors = pd.concat([reported_errors, pd.DataFrame([{
                'model_name': model_name,
                'error': f'VRAM_HIGH_BEFORE_LOAD ({vram_after_clean:.1f}%)'
            }])], ignore_index=True)
            continue

    print(f'⏳ Cargando modelo {model_name}')

    model = None

    try:
        # Obtener información del modelo
        try:
            info = model_info(model_name)
            langs = info.cardData.get("languages") or info.cardData.get("language")
            mode = 'multilingual' if len(langs) > 1 else 'monolingual'
        except:
            mode = 'unknown'

        # Descarga local (esto no consume tanta RAM)
        print(f'📥 Descargando {model_name}...')
        local_dir = snapshot_download(
            repo_id=model_name,
            cache_dir=cache_dir,
            use_auth_token=True,
            resume_download=True,
            force_download=False,
            local_files_only=False
        )

        # Cargar modelo CON monitoreo de RAM y VRAM
        print(f'🔄 Cargando en memoria y monitoreando recursos...')
        result = load_model_with_monitoring(
            model_name,
            local_dir,
            device,
            RAM_THRESHOLD,
            VRAM_THRESHOLD,
            RAM_CHECK_INTERVAL
        )

        model, num_params, error_msg, ram_val, vram_val = result

        if error_msg:
          print('Carga fallida. Intendando descargar modelo directamente en memoria...')
          result = load_model_with_monitoring(
              model_name,
              model_name, # directamente el modelo
              device,
              RAM_THRESHOLD,
              VRAM_THRESHOLD,
              RAM_CHECK_INTERVAL
          )

          model, num_params, error_msg, ram_val, vram_val = result

        if model is None:
          raise MemoryError(f"Carga fallida: {e} (RAM: {ram_val:.1f}%, VRAM: {vram_val:.1f}%)")


        # Evaluación
        print(f'🧪 Evaluando modelo...')
        evaluation_results = spanish_monolingual_evaluation(model, device, source)

        # Guardar resultados
        df = pd.concat([df, pd.DataFrame([{
            'model_name': model_name,
            'num_params': num_params,
            'languages': mode,
            'map_es_es': evaluation_results.get('map', np.nan),
            'mrr': evaluation_results.get('mrr', np.nan),
            'ndcg': evaluation_results.get('ndcg', np.nan),
            'p@5': evaluation_results.get('precision@5', np.nan),
            'p@10': evaluation_results.get('precision@10', np.nan),
            'p@100': evaluation_results.get('precision@100', np.nan)
        }])], ignore_index=True)

        print(f'✅ Modelo {model_name} evaluado exitosamente. MAP: {evaluation_results.get('map', np.nan)}')

    except MemoryError as e:
        error_detail = str(e)
        print(f'❌ Error de memoria con {model_name}: {error_detail}')
        errors.append(model_name)
        reported_errors = pd.concat([reported_errors, pd.DataFrame([{
            'model_name': model_name,
            'error': error_detail
        }])], ignore_index=True)

    except Exception as e:
        error_detail = str(e)[:200]
        print(f'❌ Error al procesar {model_name}: {error_detail}')
        errors.append(model_name)
        reported_errors = pd.concat([reported_errors, pd.DataFrame([{
            'model_name': model_name,
            'error': error_detail
        }])], ignore_index=True)

    finally:
        # SIEMPRE limpiar memoria
        if model is not None:
            del model
        clear_memory()
        time.sleep(0.5)  # Pequeña pausa para asegurar limpieza

        ram_after_clear = get_ram_usage()
        vram_after_clear = get_vram_usage()
        print(f'🧹 Después de limpieza -> RAM: {ram_after_clear:.1f}% | VRAM: {vram_after_clear:.1f}%')
        print('-' * 60)

        # Borrar modelo de la cache
        if index % 3 == 0:
          for item in os.listdir(cache_dir):
            item_path = os.path.join(cache_dir, item)

            # Si es un directorio, se elimina recursivamente
            if os.path.isdir(item_path):
                shutil.rmtree(item_path)
            # Si es un archivo, se elimina directamente
            elif os.path.isfile(item_path):
                os.remove(item_path)

    # Guardar checkpoint
    if index % 1 == 0:
        df.to_csv(output_dir / 'mteb_evaluation_results_checkpoint.csv', index=False)
        reported_errors.to_csv(output_dir / 'mteb_reported_errors_checkpoint.csv', index=False)
        with open(output_dir / "mteb_errors_checkpoint.txt", "w") as archivo:
            for item in errors:
                archivo.write(f"{item}\n")
        print('=' * 60)
        print(f'💾 Checkpoint guardado en iteracion {index}')
        print(f'📊 Modelos evaluados: {len(df)} | Errores: {len(errors)}')
        print('=' * 60)

print('\n🎉 Evaluación completada!')
print(f'✅ Modelos evaluados: {len(df)}')
print(f'❌ Errores: {len(errors)}')
print(f'📋 Detalles de errores guardados en reported_errors')

🚀 Iniciando evaluación con control de RAM y VRAM
📊 Umbral RAM: 90% | Umbral VRAM: 95%
💾 RAM inicial: 44.8%
🎮 CUDA disponible: Tesla T4
🎮 VRAM total: 14.7 GB
🎮 VRAM inicial: 0.0% (0.00 GB)
Continuamos con iteracion 87.
📊 Estado inicial -> RAM: 44.8% | VRAM: 0.0% (0.00 GB)
⏳ Cargando modelo Alibaba-NLP/gte-Qwen2-1.5B-instruct
📥 Descargando Alibaba-NLP/gte-Qwen2-1.5B-instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

   ✓ Modelo cargado en 55.7s
   📊 Post-carga -> RAM: 63.4% | VRAM: 35.1% (5.17 GB)
🧪 Evaluando modelo...
Loading Spanish data...
Encoding data: mstsb-paraphrase-multilingual-mpnet-base-v2 on device: cuda
Calculating similarities and preparing results...
Evaluating Spanish monolingual performance...


/usr/local/lib/python3.12/dist-packages/ranx/metrics/average_precision.py:49: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  scores[i] = _average_precision(qrels[i], run[i], k, rel_lvl)


In [ ]:
df = df.sort_values('map_es_es', ascending=False).reset_index(drop=True)
df.to_csv(output_dir / 'mteb_evaluation_results.csv', index=False)

# los modelos que no se han podido ejecutar es porque son cuantizaciones de modelos mayores que si se han ejecutado
with open(output_dir / "mteb_errors.txt", "w") as archivo:
    for item in errors:
        archivo.write(f"{item}\n")

## Spanish models iteration

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
source = 'validation'

with open(project_dir / 'models' / 'results' / 'spanish' / 'es_models_list.txt', 'r') as file:
    models = file.read().splitlines()

df = pd.DataFrame(columns=['model_name', 'num_params', 'languages', 'map_es_es', 'mrr', 'ndcg', 'p@5', 'p@10', 'p@100'])
errors = []

In [ ]:
for index, model_name in enumerate(models):

  if model_name in df.model_name.values or model_name in errors:
    continue

  if index == 0:
    print('Comenzamos desde el principio')
  else:
    print(f'Continuamos con Iteracion {index}.')

  try:
    info = model_info(model_name)
    langs = info.cardData.get("languages") or info.cardData.get("language")
    mode = 'multilingual' if len(langs) > 1 else 'monolingual'
  except:
    mode = 'monolingual'

  try:
    model = SentenceTransformer(model_name, device=device, trust_remote_code=True)
    evaluation_results = spanish_monolingual_evaluation(model, device, source)
  except:
    errors.append(model_name)
    continue

  num_params = sum(p.numel() for p in model.parameters())

  df = pd.concat([df, pd.DataFrame([{
        'model_name': model_name,
        'num_params': num_params,
        'languages': mode,
        'map_es_es': evaluation_results.get('map', np.nan),
        'mrr': evaluation_results.get('mrr', np.nan),
        'ndcg': evaluation_results.get('ndcg', np.nan),
        'p@5': evaluation_results.get('precision@5', np.nan),
        'p@10': evaluation_results.get('precision@10', np.nan),
        'p@100': evaluation_results.get('precision@100', np.nan)
  }])], ignore_index=True)


  if index % 2 == 0:
    df.to_csv(output_dir / 'es_evaluation_results_checkpoint.csv', index=False)
    with open(output_dir / "es_errors_checkpoint.txt", "w") as archivo:
        for item in errors:
            archivo.write(f"{item}\n")

    print('==========================================')
    print(f'Checkpoint en iteracion {index}')
    print('==========================================')

Continuamos con Iteracion 86.


Continuamos con Iteracion 87.


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at bertin-project/bertin-roberta-base-spanish and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/pytorch_model.bin:   0%|          | 0.00/789k [00:00<?, ?B/s]

Loading Spanish data...
Encoding data: Model_dccuchile_bert-base-spanish-wwm-uncased_100_Epochs on device: cuda
Calculating similarities and preparing results...
Evaluating Spanish monolingual performance...
Spanish evaluation completed
Checkpoint en iteracion 88


In [ ]:
df = df.sort_values('map_es_es', ascending=False).reset_index(drop=True)
df.to_csv(output_dir / 'es_evaluation_results.csv', index=False)

# los modelos que no se han podido ejecutar es porque son cuantizaciones de modelos mayores que si se han ejecutado
with open(output_dir / "es_errors.txt", "w") as archivo:
    for item in errors:
        archivo.write(f"{item}\n")

In [ ]:
df

,model_name,num_params,languages,map_es_es,mrr,ndcg,p@5,p@10,p@100
0,PlanTL-GOB-ES/Controversy-Prediction,124643328,monolingual,0.163781,0.458191,0.537438,0.393514,0.356757,0.106595
1,PlanTL-GOB-ES/RoBERTalex,125978112,monolingual,0.106082,0.405401,0.480093,0.302703,0.237838,0.071243
2,PlanTL-GOB-ES/bsc-bio-ehr-es,124643328,monolingual,0.121292,0.423240,0.500913,0.343784,0.290811,0.089189
3,PlanTL-GOB-ES/bsc-bio-ehr-es-cantemist,124643328,monolingual,0.146873,0.454792,0.531166,0.395676,0.342162,0.107568
4,PlanTL-GOB-ES/bsc-bio-ehr-es-pharmaconer,124643328,monolingual,0.129205,0.438381,0.514391,0.367568,0.318919,0.096486
...,...,...,...,...,...,...,...,...,...
80,dccuchile/tulio-chilean-spanish-bert,109850880,monolingual,0.199765,0.474575,0.577699,0.445405,0.365405,0.132054
81,hiiamsid/sentence_similarity_spanish_es,109850880,monolingual,0.359950,0.543644,0.696632,0.614054,0.571351,0.209568
82,hackathon-pln-es/paraphrase-spanish-distilroberta,124645632,monolingual,0.306656,0.506510,0.644225,0.517838,0.489730,0.193946
83,jinaai/jina-embeddings-v2-base-es,160850688,multilingual,0.384104,0.560721,0.708983,0.622703,0.588108,0.215676


## Intento de evaluación de errores

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
source = 'validation'

with open(project_dir / 'models' / 'results' / 'mteb' / 'mteb_errors.txt', 'r') as file:
    models = file.read().splitlines()

models = [model for model in models if model != '']

combined = pd.read_csv(project_dir / 'models' / 'results' / 'evaluation_results.csv')

df = pd.DataFrame(columns=['model_name', 'num_params', 'languages', 'map_es_es', 'mrr', 'ndcg', 'p@5', 'p@10', 'p@100'])
reported_errors = pd.DataFrame(columns=['model_name', 'error'])

errors = []

In [ ]:
df = pd.read_csv('/content/mteb_evaluation_results_checkpoint.csv')
reported_errors = pd.read_csv('/content/mteb_reported_errors_checkpoint.csv')

with open('/content/mteb_errors_checkpoint.txt', "r") as archivo:
    errors = [line.strip() for line in archivo.readlines()]

In [ ]:
df

,model_name,num_params,languages,map_es_es,mrr,ndcg,p@5,p@10,p@100
0,jinaai/jina-embeddings-v3,572310396,multilingual,0.445291,0.547297,0.740716,0.656216,0.615135,0.241892
1,intfloat/multilingual-e5-small,117653760,multilingual,0.351316,0.539189,0.688664,0.608649,0.584865,0.197405
2,intfloat/multilingual-e5-large-instruct,559890432,multilingual,0.442597,0.538243,0.739878,0.658378,0.622162,0.242108
3,intfloat/multilingual-e5-large,559890432,multilingual,0.338105,0.543115,0.683437,0.581622,0.548649,0.197514
4,intfloat/multilingual-e5-base,278043648,multilingual,0.346278,0.539189,0.686449,0.590270,0.559459,0.198432


In [ ]:
errors

['nvidia/NV-Embed-v2',
 'nvidia/NV-Embed-v1',
 'minishlab/potion-multilingual-128M',
 'jinaai/jina-embeddings-v4',
 'intfloat/e5-mistral-7b-instruct']

In [ ]:
# Configuración
RAM_THRESHOLD = 90  # Ajusta según tu entorno (80-85 para Colab)
VRAM_THRESHOLD = 95  # Umbral para VRAM (puede ser más alto que RAM)
RAM_CHECK_INTERVAL = 0.3  # Verificar cada 0.5 segundos

def get_ram_usage():
    """Retorna el porcentaje de RAM utilizado"""
    return psutil.virtual_memory().percent

def get_vram_usage():
    """Retorna el porcentaje de VRAM utilizado"""
    if torch.cuda.is_available():
        total_vram = torch.cuda.get_device_properties(0).total_memory
        allocated_vram = torch.cuda.memory_allocated(0)
        return (allocated_vram / total_vram) * 100
    return 0

def get_vram_gb():
    """Retorna VRAM usado en GB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated(0) / 1024**3
    return 0

def clear_memory():
    """Libera memoria de forma agresiva"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def load_model_with_monitoring(model_name, local_dir, device, ram_threshold=85, vram_threshold=90, check_interval=2.0):
    """
    Carga un modelo monitoreando RAM y VRAM periódicamente durante la carga.

    Args:
        model_name: Nombre del modelo
        local_dir: Directorio local del modelo
        device: Dispositivo (cuda/cpu)
        ram_threshold: Umbral de RAM en porcentaje
        vram_threshold: Umbral de VRAM en porcentaje
        check_interval: Intervalo de verificación en segundos

    Returns:
        tuple: (model, num_params, error_message)
    """
    try:
        print(f'   📦 Iniciando carga del modelo...')

        # Verificar memoria antes de empezar
        ram_initial = get_ram_usage()
        vram_initial = get_vram_usage()
        print(f'   📊 Pre-carga -> RAM: {ram_initial:.1f}% | VRAM: {vram_initial:.1f}%')

        if ram_initial > ram_threshold:
            return None, None, f"RAM_TOO_HIGH_BEFORE_LOAD", ram_initial, vram_initial

        if vram_initial > vram_threshold:
            return None, None, f"VRAM_TOO_HIGH_BEFORE_LOAD", ram_initial, vram_initial

        # Cargar el modelo (SIN subprocess, para evitar problemas de serialización)
        start_time = time.time()
        model = SentenceTransformer(local_dir, device=device, trust_remote_code=True, model_kwargs={"torch_dtype": torch.float16, "low_cpu_mem_usage": True})
        load_time = time.time() - start_time

        # Verificar memoria después de cargar
        ram_after = get_ram_usage()
        vram_after = get_vram_usage()
        vram_gb = get_vram_gb()

        print(f'   ✓ Modelo cargado en {load_time:.1f}s')
        print(f'   📊 Post-carga -> RAM: {ram_after:.1f}% | VRAM: {vram_after:.1f}% ({vram_gb:.2f} GB)')

        # Verificar si se excedieron los umbrales DESPUÉS de cargar
        if ram_after > ram_threshold:
            print(f'   ⚠️  RAM excedió umbral después de cargar: {ram_after:.1f}% > {ram_threshold}%')
            del model
            clear_memory()
            return None, None, f"RAM_EXCEEDED_AFTER_LOAD", ram_after, vram_after

        if vram_after > vram_threshold:
            print(f'   ⚠️  VRAM excedió umbral después de cargar: {vram_after:.1f}% > {vram_threshold}%')
            del model
            clear_memory()
            return None, None, f"VRAM_EXCEEDED_AFTER_LOAD", ram_after, vram_after

        # Obtener número de parámetros
        num_params = sum(p.numel() for p in model.parameters())

        return model, num_params, None, ram_after, vram_after

    except RuntimeError as e:
        if "out of memory" in str(e).lower() or "cuda" in str(e).lower():
            ram_curr = get_ram_usage()
            vram_curr = get_vram_usage()
            print(f'   ❌ Error de memoria CUDA: {e}')
            print(f'   📊 Al fallar -> RAM: {ram_curr:.1f}% | VRAM: {vram_curr:.1f}%')
            clear_memory()
            return None, None, f"CUDA_OUT_OF_MEMORY", ram_curr, vram_curr
        else:
            raise

    except Exception as e:
        ram_curr = get_ram_usage()
        vram_curr = get_vram_usage()
        print(f'   ❌ Error inesperado: {str(e)[:100]}')
        return None, None, str(e)[:200], ram_curr, vram_curr

In [ ]:
cache_dir = "/content/hf_cache"

print(f'🚀 Iniciando evaluación con control de RAM y VRAM')
print(f'📊 Umbral RAM: {RAM_THRESHOLD}% | Umbral VRAM: {VRAM_THRESHOLD}%')
print(f'💾 RAM inicial: {get_ram_usage():.1f}%')
if torch.cuda.is_available():
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'🎮 CUDA disponible: {torch.cuda.get_device_name(0)}')
    print(f'🎮 VRAM total: {total_vram_gb:.1f} GB')
    print(f'🎮 VRAM inicial: {get_vram_usage():.1f}% ({get_vram_gb():.2f} GB)')
print('=' * 60)

for index, model_name in enumerate(models):
    if model_name in df.model_name.values or model_name in errors or model_name in combined.model_name.values:
        continue

    if index == 0:
        print('Comenzamos desde el principio')
    else:
        print(f'Continuamos con iteracion {index}.')

    # Estado de memoria antes de empezar
    ram_before = get_ram_usage()
    vram_before = get_vram_usage()
    vram_gb_before = get_vram_gb()
    print(f'📊 Estado inicial -> RAM: {ram_before:.1f}% | VRAM: {vram_before:.1f}% ({vram_gb_before:.2f} GB)')

    # Verificación preventiva
    if ram_before > RAM_THRESHOLD or vram_before > VRAM_THRESHOLD:
        print(f'⚠️  Memoria demasiado alta antes de cargar - Limpiando...')
        clear_memory()
        time.sleep(1)  # Dar tiempo a que se libere
        ram_after_clean = get_ram_usage()
        vram_after_clean = get_vram_usage()
        print(f'📊 Después de limpieza -> RAM: {ram_after_clean:.1f}% | VRAM: {vram_after_clean:.1f}%')

        if ram_after_clean > RAM_THRESHOLD:
            print(f'❌ No se pudo liberar suficiente RAM ({ram_after_clean:.1f}% > {RAM_THRESHOLD}%) - Saltando {model_name}')
            errors.append(model_name)
            reported_errors = pd.concat([reported_errors, pd.DataFrame([{
                'model_name': model_name,
                'error': f'RAM_HIGH_BEFORE_LOAD ({ram_after_clean:.1f}%)'
            }])], ignore_index=True)
            continue

        if vram_after_clean > VRAM_THRESHOLD:
            print(f'❌ No se pudo liberar suficiente VRAM ({vram_after_clean:.1f}% > {VRAM_THRESHOLD}%) - Saltando {model_name}')
            errors.append(model_name)
            reported_errors = pd.concat([reported_errors, pd.DataFrame([{
                'model_name': model_name,
                'error': f'VRAM_HIGH_BEFORE_LOAD ({vram_after_clean:.1f}%)'
            }])], ignore_index=True)
            continue

    print(f'⏳ Cargando modelo {model_name}')

    model = None

    try:
        # Obtener información del modelo
        try:
            info = model_info(model_name)
            langs = info.cardData.get("languages") or info.cardData.get("language")
            mode = 'multilingual' if len(langs) > 1 else 'monolingual'
        except:
            mode = 'unknown'

        # Descarga local (esto no consume tanta RAM)
        print(f'📥 Descargando {model_name}...')
        local_dir = snapshot_download(
            repo_id=model_name,
            cache_dir=cache_dir,
            use_auth_token=True,
            resume_download=True,
            force_download=False,
            local_files_only=False
        )

        # Cargar modelo CON monitoreo de RAM y VRAM
        print(f'🔄 Cargando en memoria y monitoreando recursos...')
        result = load_model_with_monitoring(
            model_name,
            local_dir,
            device,
            RAM_THRESHOLD,
            VRAM_THRESHOLD,
            RAM_CHECK_INTERVAL
        )

        model, num_params, error_msg, ram_val, vram_val = result

        if error_msg:
          print('Carga fallida. Intendando descargar modelo directamente en memoria...')
          result = load_model_with_monitoring(
              model_name,
              model_name, # directamente el modelo
              device,
              RAM_THRESHOLD,
              VRAM_THRESHOLD,
              RAM_CHECK_INTERVAL
          )

          model, num_params, error_msg, ram_val, vram_val = result

        if model is None:
          raise MemoryError(f"Carga fallida: {e} (RAM: {ram_val:.1f}%, VRAM: {vram_val:.1f}%)")


        # Evaluación
        print(f'🧪 Evaluando modelo...')
        evaluation_results = spanish_monolingual_evaluation(model, device, source)

        # Guardar resultados
        df = pd.concat([df, pd.DataFrame([{
            'model_name': model_name,
            'num_params': num_params,
            'languages': mode,
            'map_es_es': evaluation_results.get('map', np.nan),
            'mrr': evaluation_results.get('mrr', np.nan),
            'ndcg': evaluation_results.get('ndcg', np.nan),
            'p@5': evaluation_results.get('precision@5', np.nan),
            'p@10': evaluation_results.get('precision@10', np.nan),
            'p@100': evaluation_results.get('precision@100', np.nan)
        }])], ignore_index=True)

        print(f'✅ Modelo {model_name} evaluado exitosamente. MAP: {evaluation_results.get('map', np.nan)}')

    except MemoryError as e:
        error_detail = str(e)
        print(f'❌ Error de memoria con {model_name}: {error_detail}')
        errors.append(model_name)
        reported_errors = pd.concat([reported_errors, pd.DataFrame([{
            'model_name': model_name,
            'error': error_detail
        }])], ignore_index=True)

    except Exception as e:
        error_detail = str(e)[:200]
        print(f'❌ Error al procesar {model_name}: {error_detail}')
        errors.append(model_name)
        reported_errors = pd.concat([reported_errors, pd.DataFrame([{
            'model_name': model_name,
            'error': error_detail
        }])], ignore_index=True)

    finally:
        # SIEMPRE limpiar memoria
        if model is not None:
            del model
        clear_memory()
        time.sleep(0.5)  # Pequeña pausa para asegurar limpieza

        ram_after_clear = get_ram_usage()
        vram_after_clear = get_vram_usage()
        print(f'🧹 Después de limpieza -> RAM: {ram_after_clear:.1f}% | VRAM: {vram_after_clear:.1f}%')
        print('-' * 60)

        # Borrar modelo de la cache
        if index % 3 == 0:
          for item in os.listdir(cache_dir):
            item_path = os.path.join(cache_dir, item)

            # Si es un directorio, se elimina recursivamente
            if os.path.isdir(item_path):
                shutil.rmtree(item_path)
            # Si es un archivo, se elimina directamente
            elif os.path.isfile(item_path):
                os.remove(item_path)

    # Guardar checkpoint
    if index % 1 == 0:
        df.to_csv(output_dir / 'mteb_evaluation_results_checkpoint.csv', index=False)
        reported_errors.to_csv(output_dir / 'mteb_reported_errors_checkpoint.csv', index=False)
        with open(output_dir / "mteb_errors_checkpoint.txt", "w") as archivo:
            for item in errors:
                archivo.write(f"{item}\n")
        print('=' * 60)
        print(f'💾 Checkpoint guardado en iteracion {index}')
        print(f'📊 Modelos evaluados: {len(df)} | Errores: {len(errors)}')
        print('=' * 60)

print('\n🎉 Evaluación completada!')
print(f'✅ Modelos evaluados: {len(df)}')
print(f'❌ Errores: {len(errors)}')
print(f'📋 Detalles de errores guardados en reported_errors')

🚀 Iniciando evaluación con control de RAM y VRAM
📊 Umbral RAM: 90% | Umbral VRAM: 95%
💾 RAM inicial: 5.1%
Comenzamos desde el principio
📊 Estado inicial -> RAM: 5.1% | VRAM: 0.0% (0.00 GB)
⏳ Cargando modelo sentence-transformers/static-similarity-mrl-multilingual-v1
📥 Descargando sentence-transformers/static-similarity-mrl-multilingual-v1...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

🔄 Cargando en memoria y monitoreando recursos...
   📦 Iniciando carga del modelo...
   📊 Pre-carga -> RAM: 12.5% | VRAM: 0.0%


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

   ✓ Modelo cargado en 10.2s
   📊 Post-carga -> RAM: 43.1% | VRAM: 0.0% (0.00 GB)
🧪 Evaluando modelo...
Loading Spanish data...
Encoding data: 8f1be76584d7beaa6ed08419872ef7454de8e88f on device: cpu


In [ ]:
df = df.sort_values('map_es_es', ascending=False).reset_index(drop=True)
df.to_csv(output_dir / 'results_from_errors.csv', index=False)

reported_errors.to_csv(output_dir / 'reported_errors.csv', index=False)

with open(output_dir / "mteb_errors.txt", "w") as archivo:
    for item in errors:
        archivo.write(f"{item}\n")